# Simplified Baseline Experiments - Fast & Interpretable

**Objective**: Compare simple, fast baseline models for sentiment analysis
- Logistic Regression (classification)
- Logistic Regression + Weighted Average (regression variant)
- Linear Regression + Nearest Label Assignment (regression variant)
- Naive Bayes (fast baseline)
- Linear SVC

**Evaluation Metric**: `1 - MAE/4` (competition metric)

**Key Focus**: 
- Use pre-detected languages stored in dataset
- Provide deeper analysis per model
- Fast execution on macOS
- Insights for future BERT/augmentation strategies

## Section 1: Import Required Libraries

In [1]:
import sys
print(sys.executable)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import json


# Add baselines module to path
sys.path.insert(0, str(Path.cwd()))

from src.preprocessing import PreprocessingFactory, LanguageDetector
from src.models import (
    BOWLogisticRegression, BOWLogisticRegressionWeighted, 
    BOWLinearRegression, BOWNaiveBayes, BOWLinearSVC,
    create_model_from_config
)
from src.utils import MetricsComputer, ExperimentLogger, DataAnalyzer
from src.experiments import ExperimentRunner, BaselineComparison

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully")

/Users/aureliencormoreche/Documents/ETH_S26/CIL/CIL_sentiment_analysis/text_classifier/.venv/bin/python
✓ All libraries imported successfully


## Section 2: Load and Prepare Data

In [2]:
# Load data with pre-detected languages
data_dir = Path('../data')

# Try to load data with language information
try:
    train_full = pd.read_csv(data_dir / 'train_w_language.csv')
    test_full = pd.read_csv(data_dir / 'test_w_language.csv')
    print("✓ Loaded data with pre-detected languages")
except FileNotFoundError:
    print("⚠ Language data not found, loading base data")
    train_full = pd.read_csv(data_dir / 'train.csv')
    test_full = pd.read_csv(data_dir / 'test.csv')
    
    # Detect languages
    print("Detecting languages...")
    detector = LanguageDetector()
    train_full['language'] = LanguageDetector.detect_language_batch(train_full)
    test_full['language'] = LanguageDetector.detect_language_batch(test_full)



print(f"\nTrain set size: {len(train_full)}")
print(f"Test set size: {len(test_full)}")
print(f"\nTrain label distribution:\n{train_full['label'].value_counts().sort_index()}")
print(f"\nTrain language distribution:\n{train_full['language'].value_counts()}")
print(f"\nFirst 3 samples:")
print(train_full.head(3))

✓ Loaded data with pre-detected languages

Train set size: 252000
Test set size: 168000

Train label distribution:
label
0    50400
1    50400
2    50400
3    50400
4    50400
Name: count, dtype: int64

Train language distribution:
language
en    126073
de    125927
Name: count, dtype: int64

First 3 samples:
   Unnamed: 0  id                                           sentence  label  \
0           0   0  moderner weihnachtsbaum in weißdieses jahr wol...      4   
1           1   1  passt wie angegossentasche kam schnell und gut...      4   
2           2   2  schlechte qualitätich habe sehr lange auf dies...      1   

  language  
0       de  
1       de  
2       de  


In [3]:
# Create train/validation split (90/10) with stratification
train_df, val_df = train_test_split(
    train_full, 
    test_size=0.1, 
    stratify=train_full['label'], 
    random_state=42
)

print(f"Train: {len(train_df)}, Validation: {len(val_df)}")
print(f"Train label distribution:\n{train_df['label'].value_counts().sort_index()}")

# Create output directory for results
output_dir = Path('outputs_v2')
output_dir.mkdir(exist_ok=True)
print(f"\n✓ Output directory: {output_dir}")

Train: 226800, Validation: 25200
Train label distribution:
label
0    45360
1    45360
2    45360
3    45360
4    45360
Name: count, dtype: int64

✓ Output directory: outputs_v2


## Section 3: Compare Baseline Model Types

In [4]:
# Run comprehensive baseline comparison
comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'model_types')
)

model_results = comparison.compare_model_types()

print("\n" + "="*70)
print("MODEL TYPE COMPARISON RESULTS")
print("="*70)

# Create results summary
model_summary = []
for result in model_results:
    config_name = result['config']['name']
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    accuracy = result['val_metrics'].get('accuracy', 'N/A')
    model_summary.append({
        'Model': config_name,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
        'Accuracy': f"{accuracy:.4f}" if accuracy != 'N/A' else accuracy,
    })

model_df = pd.DataFrame(model_summary)
print(model_df.to_string(index=False))

# Find best model
best_idx = np.argmin([r['val_metrics']['mae'] for r in model_results])
best_model = model_results[best_idx]
best_name = best_model['config']['name']
print(f"\n→ Best Model: {best_name} (MAE: {best_model['val_metrics']['mae']:.4f})")


COMPARING BASELINE MODEL TYPES


Running experiments:   0%|          | 0/2 [00:00<?, ?it/s]


Running: baseline_xgboost
  Train MAE: 0.6006, Val MAE: 0.6663, Score: 0.8334


Running experiments:  50%|█████     | 1/2 [22:33<22:33, 1353.43s/it]


Running: baseline_random_forest
  Train MAE: 0.9375, Val MAE: 0.9795, Score: 0.7551


Running experiments: 100%|██████████| 2/2 [33:19<00:00, 999.65s/it] 

Results saved to outputs_v2/model_types/model_types/results.csv
Full results saved to outputs_v2/model_types/model_types/results.json

MODEL TYPE COMPARISON RESULTS
                 Model Val MAE Metric (1-MAE/4) Accuracy
      baseline_xgboost  0.6663           0.8334   0.5356
baseline_random_forest  0.9795           0.7551   0.4391

→ Best Model: baseline_xgboost (MAE: 0.6663)


## Section 4: Deep Analysis of Best Model

In [ ]:
# Extract predictions from best model
best_model = model_results[best_idx]
y_val_true = val_df['label'].values
y_val_pred = best_model['val_preds']

# Compute detailed error statistics
error_stats = DataAnalyzer.compute_error_statistics(y_val_true, y_val_pred)

print("\n" + "="*70)
print(f"DETAILED ANALYSIS: {best_name}")
print("="*70)
print(f"\nError Statistics:")
print(f"  Mean Error: {error_stats['mean_error']:.4f}")
print(f"  Median Error: {error_stats['median_error']:.4f}")
print(f"  Std Error: {error_stats['std_error']:.4f}")
print(f"  Max Error: {error_stats['max_error']}")
print(f"  Perfect Predictions: {error_stats['perfect_predictions']} ({100*error_stats['perfect_predictions']/len(y_val_true):.1f}%)")
print(f"  Large Errors (>=2): {error_stats['large_errors']} ({100*error_stats['large_errors']/len(y_val_true):.1f}%)")

print(f"\nError Distribution:")
for error_key, error_data in error_stats['error_distribution'].items():
    print(f"  {error_key}: {error_data['count']:4d} ({error_data['percentage']:5.1f}%)")

# Per-label metrics
label_metrics = DataAnalyzer.compute_label_distribution_metrics(y_val_true, y_val_pred)
print(f"\nPer-Label Performance:")
for label_key, metrics in label_metrics.items():
    print(f"  {label_key}: MAE={metrics['mae']:.4f}, Accuracy={metrics['accuracy']:.4f}, Count={metrics['count']}")

# Language performance (if available)
if 'language' in val_df.columns:
    lang_metrics = DataAnalyzer.analyze_language_performance(val_df, y_val_true, y_val_pred)
    print(f"\nPer-Language Performance:")
    for lang_key, metrics in lang_metrics.items():
        print(f"  {lang_key}: MAE={metrics['mae']:.4f}, Accuracy={metrics['accuracy']:.4f}, Count={metrics['count']}")

In [ ]:
# Visualize predictions and errors
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Error distribution histogram
errors = np.abs(y_val_true - y_val_pred)
axes[0, 0].hist(errors, bins=range(0, int(errors.max())+2), edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 0].set_xlabel('Absolute Error')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title(f'Error Distribution - {best_name}')
axes[0, 0].axvline(errors.mean(), color='r', linestyle='--', linewidth=2, label=f'Mean: {errors.mean():.2f}')
axes[0, 0].legend()

# Plot 2: Predictions vs Ground Truth
axes[0, 1].scatter(y_val_true, y_val_pred, alpha=0.4, s=20, color='darkblue')
axes[0, 1].plot([0.5, 5.5], [0.5, 5.5], 'r--', lw=2, label='Perfect Predictions')
axes[0, 1].set_xlabel('Ground Truth')
axes[0, 1].set_ylabel('Predictions')
axes[0, 1].set_title('Predictions vs Ground Truth')
axes[0, 1].set_xlim(0.5, 5.5)
axes[0, 1].set_ylim(0.5, 5.5)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Label-wise MAE
labels = sorted(np.unique(y_val_true))
label_maes = []
for label in labels:
    mask = y_val_true == label
    if mask.sum() > 0:
        mae = np.abs(y_val_true[mask] - y_val_pred[mask]).mean()
        label_maes.append(mae)
    else:
        label_maes.append(0)

axes[1, 0].bar([str(int(l)) for l in labels], label_maes, color='coral', edgecolor='black')
axes[1, 0].set_xlabel('Label')
axes[1, 0].set_ylabel('Mean Absolute Error')
axes[1, 0].set_title('MAE by Label')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Prediction counts by label
pred_counts = pd.Series(y_val_pred).value_counts().sort_index()
true_counts = pd.Series(y_val_true).value_counts().sort_index()

x = np.arange(len(labels))
width = 0.35
axes[1, 1].bar(x - width/2, [true_counts.get(l, 0) for l in labels], width, label='True', color='steelblue', edgecolor='black')
axes[1, 1].bar(x + width/2, [pred_counts.get(l, 0) for l in labels], width, label='Predicted', color='coral', edgecolor='black')
axes[1, 1].set_xlabel('Label')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Label Distribution: True vs Predicted')
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels([str(int(l)) for l in labels])
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'best_model_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: best_model_analysis.png")

## Section 5: Feature Extraction Comparison

In [ ]:
# Compare different feature extraction strategies
feature_comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'feature_extraction')
)

feature_results = feature_comparison.compare_feature_extraction()

print("\n" + "="*70)
print("FEATURE EXTRACTION COMPARISON")
print("="*70)

feature_summary = []
for result in feature_results:
    config_name = result['config']['name']
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    feature_summary.append({
        'Feature Strategy': config_name,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
    })

feature_df = pd.DataFrame(feature_summary)
print(feature_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
maes = [r['val_metrics']['mae'] for r in feature_results]
names = [r['config']['name'] for r in feature_results]
colors = ['green' if 'tfidf' in n else 'steelblue' for n in names]

ax.barh(names, maes, color=colors, edgecolor='black', alpha=0.7)
ax.set_xlabel('Validation MAE')
ax.set_title('Feature Extraction Strategy Comparison')
ax.invert_yaxis()
for i, (name, mae) in enumerate(zip(names, maes)):
    ax.text(mae, i, f' {mae:.4f}', va='center')

plt.tight_layout()
plt.savefig(output_dir / 'feature_extraction_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: feature_extraction_comparison.png")

## Section 6: Preprocessing Impact

In [ ]:
# Compare preprocessing strategies
prep_comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'preprocessing')
)

prep_results = prep_comparison.compare_preprocessing()

print("\n" + "="*70)
print("PREPROCESSING IMPACT ANALYSIS")
print("="*70)

prep_summary = []
for result in prep_results:
    config_name = result['config']['preprocessing']
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    prep_summary.append({
        'Preprocessing': config_name,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
    })

prep_df = pd.DataFrame(prep_summary)
print(prep_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
maes = [r['val_metrics']['mae'] for r in prep_results]
names = [r['config']['preprocessing'] for r in prep_results]
colors = ['green' if 'aggressive' in n or 'extended' in n else 'steelblue' for n in names]

ax.barh(names, maes, color=colors, edgecolor='black', alpha=0.7)
ax.set_xlabel('Validation MAE')
ax.set_title('Preprocessing Strategy Comparison')
ax.invert_yaxis()
for i, (name, mae) in enumerate(zip(names, maes)):
    ax.text(mae, i, f' {mae:.4f}', va='center')

plt.tight_layout()
plt.savefig(output_dir / 'preprocessing_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: preprocessing_comparison.png")

## Section 7: Language-Specific vs Multilingual Models

In [ ]:
# Compare language-specific vs multilingual models
lang_comparison = BaselineComparison(
    train_df=train_df,
    val_df=val_df,
    test_df=test_full,
    text_col='sentence',
    label_col='label',
    output_dir=str(output_dir / 'language_comparison')
)

lang_results = lang_comparison.compare_language_specific()

print("\n" + "="*70)
print("LANGUAGE-SPECIFIC VS MULTILINGUAL COMPARISON")
print("="*70)

lang_summary = []
for lang_key, result in lang_results.items():
    mae = result['val_metrics']['mae']
    score = result['val_metrics']['competition_score']
    count = len(result.get('val_preds', []))
    lang_summary.append({
        'Approach': lang_key,
        'Val MAE': f"{mae:.4f}",
        'Metric (1-MAE/4)': f"{score:.4f}",
    })

lang_df = pd.DataFrame(lang_summary)
print(lang_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
maes = [r['val_metrics']['mae'] for r in lang_results.values()]
names = list(lang_results.keys())
colors = ['green' if 'multilingual' in n else 'coral' for n in names]

ax.barh(names, maes, color=colors, edgecolor='black', alpha=0.7)
ax.set_xlabel('Validation MAE')
ax.set_title('Language-Specific vs Multilingual Approach')
ax.invert_yaxis()
for i, (name, mae) in enumerate(zip(names, maes)):
    ax.text(mae, i, f' {mae:.4f}', va='center')

plt.tight_layout()
plt.savefig(output_dir / 'language_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: language_comparison.png")